In [ ]:
import pandas as pd

df_full = pd.read_csv('df_with_marking_final_full.csv')

In [2]:
df_marking = df_full.sample(n=300, random_state=42).reset_index(drop=True)

In [ ]:
import re
import pandas as pd

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

conviction_keywords = {
    'нет': ['не имеющего судимости', 'ранее не судимого', 'ранее не судим', 'не судимого', 'ранее не судимой', 'несудимого', 
            'не судимый', 'не судимой', 'ранее не судима', 'судимостей, влекущих правовые последствия, не имеющего', 'судимости не имеющего'],
    'да': ['судимого', 'ранее судимого', 'ранее судим', 'ранее был осужден', 'освобожден', 'ранее судимой', 'имеющего судимость', 'с учетом непогашенных судимостей', 'судимого:', 'судимой:', 'судимой']
}

train_data_prior_convictions = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    true_label = str(row.get("predicted_prior_convictions")).strip().lower()
    if true_label not in conviction_keywords:
        continue

    found = False
    for keyword in conviction_keywords[true_label]:
        match = re.search(r'\b' + re.escape(keyword) + r'\b', text_lower)
        if match:
            start, end = match.span()
            # print(f"Найдено: '{text[start:end]}' (id={row['id']})")
            train_data_prior_convictions.append((text, {"entities": [(start, end, "PRIOR_CONVICTIONS")]}))
            found = True
            break

    if not found:
        train_data_prior_convictions.append((text, {"entities": []}))
        # print(f"Не найдены ключевые слова '{true_label}' в id={idx}")
print(f"TRAIN_DATA_PRIOR_CONVICTIONS готово: {len(train_data_prior_convictions)} примеров")

TRAIN_DATA_PRIOR_CONVICTIONS готово: 298 примеров


In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("PRIOR_CONVICTIONS")

examples = []
for text, annot in train_data_prior_convictions:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_b_prior_convictions_model")
print("Модель сохранена в 'ner_b_prior_convictions_model'")

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Epoch 1, Losses: {'ner': 259999.86196049728}
Epoch 2, Losses: {'ner': 3078.85206323687}
Epoch 3, Losses: {'ner': 190.88369156266114}
Epoch 4, Losses: {'ner': 131.2763116296534}
Epoch 5, Losses: {'ner': 105.65724392676633}
Epoch 6, Losses: {'ner': 127.03700334553842}
Epoch 7, Losses: {'ner': 106.68940602700712}
Epoch 8, Losses: {'ner': 91.08094378804465}
Epoch 9, Losses: {'ner': 87.33466269726493}
Epoch 10, Losses: {'ner': 95.35180794223241}
Epoch 11, Losses: {'ner': 83.49746689215692}
Epoch 12, Losses: {'ner': 72.68655542263357}
Epoch 13, Losses: {'ner': 69.69607826462918}
Epoch 14, Losses: {'ner': 77.93948956470577}
Epoch 15, Losses: {'ner': 65.65209410342567}
Модель сохранена в 'ner_b_prior_convictions_model'


In [11]:
df_test = pd.read_csv('df_with_marking_final.csv')

In [13]:
import spacy
from sklearn.metrics import accuracy_score, f1_score

nlp_prior = spacy.load("ner_b_prior_convictions_model")

conviction_keywords = {
    'нет': ['не имеющего судимости', 'ранее не судимого', 'ранее не судим', 'не судимого', 'ранее не судимой', 'несудимого',
            'не судимый', 'не судимой', 'ранее не судима', 'судимостей, влекущих правовые последствия, не имеющего', 'судимости не имеющего'],
    'да': ['судимого', 'ранее судимого', 'ранее судим', 'ранее был осужден', 'освобожден', 'ранее судимой', 'имеющего судимость',
           'с учетом непогашенных судимостей', 'судимого:', 'судимой:', 'судимой']
}

flat_keyword_map = {}
for label, phrases in conviction_keywords.items():
    for phrase in phrases:
        flat_keyword_map[phrase.lower()] = label

y_true = []
y_pred = []

for idx, row in df_test.iterrows():
    true_label = str(row.get("prior_convictions")).strip().lower()
    if true_label not in ["да", "нет"]:
        continue

    text = f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"
    doc = nlp_prior(text.lower()) 

    predicted_label = "нет"  

    for ent in doc.ents:
        if ent.label_ == "PRIOR_CONVICTIONS":
            ent_text = ent.text.strip().lower()
            for key_phrase, label in flat_keyword_map.items():
                if key_phrase in ent_text:
                    predicted_label = label
                    break
            break  

    y_true.append(true_label)
    y_pred.append(predicted_label)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label="да")

print(f"Accuracy: {acc:.2%}")
print(f"F1-score: {f1:.2%}")
print(f"Примеров: {len(y_true)}")

Accuracy: 81.44%
F1-score: 55.00%
Примеров: 97
